In [ ]:
from pathlib import Path

# Run this baseline for the attached ATCoder V3 Kaggle dataset.
# Add the ATCoder V3 dataset to the Kaggle notebook inputs before running all cells.
DATASET_KEYS = ("at-coder",)
RUN_LABEL = "gnn_baselines"


# Kaggle's current PyTorch build cannot execute kernels on Tesla P100 (sm_60).
# These notebooks are intended for a T4-class accelerator; two T4s are fine,
# although this single-process implementation uses GPU 0.
import os
# Prevent CUDA allocator fragmentation on long tree-baseline runs.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is enabled. In Kaggle select GPU accelerator: 2x T4.")
_GPU_CAPABILITY = torch.cuda.get_device_capability(0)
_GPU_NAME = torch.cuda.get_device_name(0)
if _GPU_CAPABILITY[0] < 7:
    raise RuntimeError(
        f"{_GPU_NAME} has unsupported CUDA capability sm_{_GPU_CAPABILITY[0]}{_GPU_CAPABILITY[1]}. "
        "Select 2x T4 in Kaggle Accelerator settings, restart the session, and Run All."
    )
print({"gpu": _GPU_NAME, "capability": f"sm_{_GPU_CAPABILITY[0]}{_GPU_CAPABILITY[1]}", "gpu_count": torch.cuda.device_count()})
# Runtime profile. Use quick_1h for preliminary results; change only this
# value to extended_6_7h for the larger follow-up run.
RUN_PROFILE = "final_full"
RUN_PRESETS = {"quick_1h": {'max_train_pairs': 30000, 'max_valid_pairs': 5000, 'max_test_pairs': 5000, 'epochs': 8, 'patience': 3}, "extended_6_7h": {'max_train_pairs': 100000, 'max_valid_pairs': 20000, 'max_test_pairs': 20000, 'epochs': 50, 'patience': 10}}
# Use this profile in every method notebook for a data-equal comparison.
RUN_PRESETS["comparison_50k"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": 50_000,
    "max_valid_pairs": 10_000,
    "max_test_pairs": 10_000,
}

# Final paper protocol: use every available pair in each official split.
RUN_PRESETS["final_full"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": None,
    "max_valid_pairs": None,
    "max_test_pairs": None,
}

# --- bounded run budget ---
# Kaggle sessions are capped, and a run that dies at the limit produces nothing.
# Training data stays large so results remain comparable with the published
# table; validation and test are capped because a bigger validation split only
# sharpens one threshold, and a bigger test split only tightens an error bar we
# do not report.
RUN_PRESETS["bounded_10h"] = {
    **RUN_PRESETS["comparison_50k"],
    "max_train_pairs": 200_000,
    "max_valid_pairs": 20_000,
    "max_test_pairs": 20_000,
}

if RUN_PROFILE not in RUN_PRESETS:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}; choose one of {tuple(RUN_PRESETS)}")
RUN_CONFIG = dict(RUN_PRESETS[RUN_PROFILE])
# Shared faithful-baseline schedule; consumed by notebook_runtime.py.
RUN_CONFIG["epochs"] = 4
RUN_CONFIG.setdefault("patience", 2)


In [ ]:
# === per-language breakdown helper ===
# Splits an already-computed set of test predictions by the language of each
# pair. No retraining and no separate per-language model: this is the same run,
# reported per language so a strong average cannot hide a collapsed language.
import gzip as _gzip
import json as _json
from pathlib import Path as _Path

import numpy as _np
import pandas as _pd

_LANGUAGE_CACHE = {}
LANGUAGE_BREAKDOWN_ROWS = []


def _resolve_codes_file():
    for root in (_Path("/kaggle/input"), _Path("/kaggle/working"), _Path(".")):
        if not root.exists():
            continue
        for name in ("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp"):
            for path in root.rglob(name):
                if path.is_file():
                    return path
    return None


def _open_any(path):
    with open(path, "rb") as probe:
        packed = probe.read(2) == b"\x1f\x8b"
    return _gzip.open(path, "rt", encoding="utf-8") if packed else open(path, "r", encoding="utf-8")


def code_languages():
    """``code_id -> language`` from the attached clean-data bundle."""
    if _LANGUAGE_CACHE:
        return _LANGUAGE_CACHE
    path = _resolve_codes_file()
    if path is None:
        print("[language-breakdown] codes.jsonl not found; breakdown will be skipped.")
        return _LANGUAGE_CACHE
    with _open_any(path) as stream:
        for line in stream:
            if not line.strip():
                continue
            record = _json.loads(line)
            code_id = str(record.get("code_id", record.get("id", record.get("idx", ""))))
            _LANGUAGE_CACHE[code_id] = str(record.get("language", record.get("lang", "unknown")))
    print(f"[language-breakdown] languages loaded for {len(_LANGUAGE_CACHE):,} codes.")
    return _LANGUAGE_CACHE


def _binary_scores(labels, predicted):
    labels = _np.asarray(labels, dtype=_np.int64)
    predicted = _np.asarray(predicted, dtype=_np.int64)
    tp = int(((predicted == 1) & (labels == 1)).sum())
    fp = int(((predicted == 1) & (labels == 0)).sum())
    tn = int(((predicted == 0) & (labels == 0)).sum())
    fn = int(((predicted == 0) & (labels == 1)).sum())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return {
        "P": precision, "R": recall, "F1": f1,
        "Acc": (tp + tn) / max(1, len(labels)),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Pairs": int(len(labels)), "Positives": int((labels == 1).sum()),
    }


def record_language_breakdown(frame, scores, threshold, *, dataset, method, graph_type=None):
    """Partition this run's test predictions by pair language and record them."""
    languages = code_languages()
    if not languages or frame is None or not len(frame):
        return []
    scores = _np.asarray(scores, dtype=_np.float64).reshape(-1)
    labels = _np.asarray(frame["label"], dtype=_np.int64).reshape(-1)
    if len(scores) != len(labels):
        print(f"[language-breakdown] skipped {method}: {len(scores)} scores vs {len(labels)} labels.")
        return []
    predicted = (scores >= float(threshold)).astype(_np.int64)

    left = [languages.get(str(value), "unknown") for value in frame["left_id"]]
    right = [languages.get(str(value), "unknown") for value in frame["right_id"]]
    # Cross-language pairs get their own bucket instead of being attributed to
    # one side; ATCoder is entirely java<->python and would otherwise vanish.
    keys = [a if a == b else f"{min(a, b)}->{max(a, b)}" for a, b in zip(left, right)]

    rows = []
    for key in sorted(set(keys)):
        mask = _np.asarray([value == key for value in keys])
        row = {"Dataset": dataset, "Method": method, "GraphType": graph_type or "", "Language": key}
        row.update(_binary_scores(labels[mask], predicted[mask]))
        row["Threshold"] = float(threshold)
        rows.append(row)
    overall = {"Dataset": dataset, "Method": method, "GraphType": graph_type or "", "Language": "ALL"}
    overall.update(_binary_scores(labels, predicted))
    overall["Threshold"] = float(threshold)
    rows.append(overall)

    LANGUAGE_BREAKDOWN_ROWS.extend(rows)
    table = _pd.DataFrame(LANGUAGE_BREAKDOWN_ROWS)
    out_path = _Path("/kaggle/working") / f"{dataset}_language_breakdown.csv"
    try:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        table.to_csv(out_path, index=False)
    except OSError:
        out_path = _Path(f"{dataset}_language_breakdown.csv")
        table.to_csv(out_path, index=False)
    print(f"\n[language-breakdown] {method}{'/' + graph_type if graph_type else ''}")
    print(_pd.DataFrame(rows)[["Language", "P", "R", "F1", "Acc", "Pairs", "Positives"]].to_string(index=False))
    print(f"[language-breakdown] written to {out_path}")
    return rows

DATASET_KEY_FOR_BREAKDOWN = "atcoder_v3"


# ATCoder V3 GNN Baselines

Kaggle-ready notebook for the full ATCoder V3 clean export.
It trains one GNN pair-classifier for each available graph type: AST, CFG, DDG, and CPG.
The notebook saves per-epoch history and plots loss/validation curves for every graph type.


## Notes

- This notebook is configured for the heavy full run over AST, CFG, DDG, and CPG.
- T4-safe adaptive batching starts at `64` and automatically halves an outlier batch after CUDA OOM.
- Outputs are saved under `/kaggle/working`: results CSV, training history CSV, and plot PNGs.


In [ ]:
# Executes the unchanged baseline pipeline once per dataset in isolated state.
# This run writes ATCoder V3-only CSV files, for example xglue4_*_results.csv.
def run_one_dataset(dataset_key: str):
    # =========================
    # Config
    # =========================
    from pathlib import Path
    import time
    import gc

    # End-to-end method runtime: loading + preprocessing + training + evaluation.
    run_started = time.perf_counter()
    DATASET_KEY = dataset_key
    KAGGLE_DATA_ROOT = Path("/kaggle/input") / DATASET_KEY
    WORK_DIR = Path("/kaggle/working")

    GRAPH_TYPES = ["ast", "cfg", "ddg", "cpg"]
# --- graph layer support ---
    UNSUPPORTED_GRAPH_LAYERS = []
    # Every language in this benchmark provides all four layers.
    GRAPH_TYPES = [layer for layer in GRAPH_TYPES if layer not in UNSUPPORTED_GRAPH_LAYERS]
    print('graph layers scored:', GRAPH_TYPES, '| dropped:', UNSUPPORTED_GRAPH_LAYERS)

    # Full dataset run: use every pair in train/valid/test.
    # Train is already close to balanced; valid/test keep the original ATCoder V3 imbalance.
    MAX_TRAIN_PAIRS = RUN_CONFIG["max_train_pairs"]
    MAX_VALID_PAIRS = RUN_CONFIG["max_valid_pairs"]
    MAX_TEST_PAIRS = RUN_CONFIG["max_test_pairs"]
    BALANCE_TRAIN = False
    BALANCE_EVAL = False
    TUNE_THRESHOLD_ON_VALID = True

    EPOCHS = RUN_CONFIG["epochs"]
    BATCH_SIZE = int(globals().get("GNN_BATCH_SIZE", 64))
    HIDDEN_DIM = 256
    EMBED_DIM = 128
    DROPOUT = 0.05
    LEARNING_RATE = 5e-4
    WEIGHT_DECAY = 1e-4
    USE_AMP = True
    THRESHOLD = 0.50
    PATIENCE = RUN_CONFIG["patience"]
    SEED = 42

    # Full graph run: use every code id that appears in the selected full pair set.
    MAX_CODE_IDS_PER_RUN = None

    USE_EIGEN_STATS = True
    USE_GRAPH_STATS = True
    DEVICE = "cuda"  # changed to cpu automatically if CUDA is unavailable

    # =========================
    # Imports and utilities
    # =========================
    import csv
    import gzip
    import zipfile
    import json
    import math
    import os
    import random
    from dataclasses import dataclass
    from typing import Dict, Iterable, List, Tuple

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from sklearn.metrics import roc_auc_score

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    if DEVICE == "cuda" and not torch.cuda.is_available():
        DEVICE = "cpu"
    print("Device:", DEVICE)

    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True


    EXTRACT_ROOT = Path(f"/kaggle/working/{DATASET_KEY}_clean_data_extracted")


    def _candidate_files_inside(directory: Path, names: tuple[str, ...]) -> list[Path]:
        files = []
        for name in names:
            files.extend([p for p in directory.rglob(name) if p.is_file()])
            # Kaggle may show zip entries created from interrupted atomic writes as e.g.
            # codes.jsonl/codes.jsonl.gz.tmp. Accept those too.
            files.extend([p for p in directory.rglob(name + ".tmp") if p.is_file()])
            files.extend([p for p in directory.rglob(name + ".gz.tmp") if p.is_file()])
        if files:
            return files

        suffixes = tuple(Path(name).suffix for name in names if Path(name).suffix)
        if suffixes:
            files.extend([
                p for p in directory.rglob("*")
                if p.is_file() and (p.suffix in suffixes or ".gz" in p.name or p.name.endswith(".tmp"))
            ])
        return files


    def _candidate_roots() -> list[Path]:
        roots = [
            KAGGLE_DATA_ROOT,
            KAGGLE_DATA_ROOT / "clean_data",
            EXTRACT_ROOT,
            EXTRACT_ROOT / "clean_data",
            Path("/kaggle/input"),
        ]
        return [root for root in roots if root.exists()]


    def _find_zip_file() -> Path | None:
        zip_names = ["Xglue.zip", "xglue.zip", "XGLUE.zip", "clean_data.zip"]
        for root in [KAGGLE_DATA_ROOT, Path("/kaggle/input")]:
            if not root.exists():
                continue
            for name in zip_names:
                direct = root / name
                if direct.is_file():
                    return direct
            zips = [p for p in root.rglob("*.zip") if p.is_file()]
            if zips:
                return sorted(zips, key=lambda p: (len(p.relative_to(root).parts), len(p.name), str(p)))[0]
        return None


    def ensure_zip_extracted() -> Path | None:
        # Accept direct data files and Kaggle's nested ``*.gz.tmp`` layout.
        if any(_candidate_files_inside(root, ("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp")) for root in _candidate_roots()):
            return None

        zip_path = _find_zip_file()
        if zip_path is None:
            return None

        marker = EXTRACT_ROOT / ".extracted_ok"
        if marker.exists():
            print("Using already extracted zip:", EXTRACT_ROOT)
            return EXTRACT_ROOT

        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        print("Extracting zip:", zip_path)
        print("Extract target:", EXTRACT_ROOT)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(EXTRACT_ROOT)
        marker.write_text(str(zip_path), encoding="utf-8")
        return EXTRACT_ROOT


    def resolve_file_path(path: Path, *fallback_names: str) -> Path:
        if path.is_file():
            return path
        if path.is_dir():
            names = fallback_names or (path.name,)
            matches = _candidate_files_inside(path, names)
            if matches:
                # Prefer the smallest nesting depth, then the shorter filename. This avoids grabbing random cache files.
                matches = sorted(matches, key=lambda p: (len(p.relative_to(path).parts), len(p.name), str(p)))
                return matches[0]
        raise FileNotFoundError(f"Expected a file but got: {path}")


    def is_gzip_file(path: Path) -> bool:
        # Trust bytes, not filename. Some Kaggle-visible files are named *.gz.tmp but are plain text.
        path = resolve_file_path(path)
        with path.open("rb") as f:
            return f.read(2) == b"\x1f\x8b"


    def open_text(path: Path):
        path = resolve_file_path(path)
        if is_gzip_file(path):
            return gzip.open(path, "rt", encoding="utf-8")
        return path.open("r", encoding="utf-8")


    def find_file(*names: str) -> Path:
        ensure_zip_extracted()
        for root in _candidate_roots():
            for name in names:
                direct = root / name
                if direct.is_file():
                    return direct
                if direct.is_dir():
                    try:
                        return resolve_file_path(direct, *names)
                    except FileNotFoundError:
                        pass
            for name in names:
                matches = [p for p in root.rglob(name) if p.is_file()]
                if matches:
                    return sorted(matches, key=lambda p: (len(p.relative_to(root).parts), len(p.name), str(p)))[0]
        available = []
        for root in _candidate_roots():
            available.extend(str(p) for p in sorted(root.rglob("*"))[:30])
        raise FileNotFoundError(
            f"Could not find any file named: {names}\n"
            f"Searched roots: {[str(r) for r in _candidate_roots()]}\n"
            f"First available paths: {available[:30]}"
        )

    codes_path = find_file("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp")
    pairs_path = find_file("pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
    graphs_path = find_file("graph_spectra.jsonl.gz", "graph_spectra.jsonl", "graph_spectra.jsonl.gz.tmp")
    print("codes:", codes_path, "is_file=", codes_path.is_file())
    print("pairs:", pairs_path, "is_file=", pairs_path.is_file())
    print("graphs:", graphs_path, "is_file=", graphs_path.is_file())


    # CodeNet injects MAX_NODES=128 in its header; other datasets use 256.
    MAX_GRAPH_NODES = int(globals().get("MAX_NODES", 256))

    # =========================
    # Load and sample pairs
    # =========================
    def load_pairs(path: Path) -> pd.DataFrame:
        path = resolve_file_path(path, "pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
        compression = "gzip" if is_gzip_file(path) else None
        df = pd.read_csv(path, compression=compression, dtype={"left_id": str, "right_id": str, "split": str, "label": np.int64})
        df["left_id"] = df["left_id"].astype(str)
        df["right_id"] = df["right_id"].astype(str)
        df["label"] = df["label"].astype(np.int64)
        return df[["split", "left_id", "right_id", "label"]]


    def sample_pairs(df: pd.DataFrame, max_pairs: int | None, *, balanced: bool, seed: int) -> pd.DataFrame:
        if max_pairs is None or len(df) <= max_pairs:
            return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        if not balanced:
            return df.sample(n=max_pairs, random_state=seed).reset_index(drop=True)

        positives = df[df.label == 1]
        negatives = df[df.label == 0]
        per_label = max_pairs // 2
        pos_sample = positives.sample(n=min(per_label, len(positives)), random_state=seed)
        neg_sample = negatives.sample(n=min(max_pairs - len(pos_sample), len(negatives)), random_state=seed + 1)
        sampled = pd.concat([pos_sample, neg_sample], ignore_index=True)
        if len(sampled) < max_pairs:
            remaining = df.drop(sampled.index, errors="ignore")
            if len(remaining):
                sampled = pd.concat(
                    [sampled, remaining.sample(n=min(max_pairs - len(sampled), len(remaining)), random_state=seed + 2)],
                    ignore_index=True,
                )
        return sampled.sample(frac=1.0, random_state=seed + 3).reset_index(drop=True)

    pairs_df = load_pairs(pairs_path)
    print("All pairs:", len(pairs_df))
    print(pairs_df.groupby(["split", "label"]).size())

    train_pairs = sample_pairs(pairs_df[pairs_df.split == "train"], MAX_TRAIN_PAIRS, balanced=BALANCE_TRAIN, seed=SEED)
    valid_pairs = sample_pairs(pairs_df[pairs_df.split == "valid"], MAX_VALID_PAIRS, balanced=BALANCE_EVAL, seed=SEED)
    test_pairs = sample_pairs(pairs_df[pairs_df.split == "test"], MAX_TEST_PAIRS, balanced=BALANCE_EVAL, seed=SEED)

    selected_ids = set(train_pairs.left_id) | set(train_pairs.right_id) | set(valid_pairs.left_id) | set(valid_pairs.right_id) | set(test_pairs.left_id) | set(test_pairs.right_id)
    if MAX_CODE_IDS_PER_RUN is not None and len(selected_ids) > MAX_CODE_IDS_PER_RUN:
        selected_ids = set(random.sample(sorted(selected_ids), MAX_CODE_IDS_PER_RUN))
        train_pairs = train_pairs[train_pairs.left_id.isin(selected_ids) & train_pairs.right_id.isin(selected_ids)].reset_index(drop=True)
        valid_pairs = valid_pairs[valid_pairs.left_id.isin(selected_ids) & valid_pairs.right_id.isin(selected_ids)].reset_index(drop=True)
        test_pairs = test_pairs[test_pairs.left_id.isin(selected_ids) & test_pairs.right_id.isin(selected_ids)].reset_index(drop=True)

    print("Selected code ids:", len(selected_ids))
    print("Train/valid/test:", len(train_pairs), len(valid_pairs), len(test_pairs))
    print("Train labels:")
    print(train_pairs.label.value_counts())
    print("Valid labels:")
    print(valid_pairs.label.value_counts())
    print("Test labels:")
    print(test_pairs.label.value_counts())


    # =========================
    # Graph loading
    # =========================
    @dataclass
    class GraphData:
        x: torch.Tensor
        edge_index: torch.Tensor
        eigen_stats: torch.Tensor
        graph_stats: torch.Tensor
        code_id: str


    def eigen_stats(values: list) -> np.ndarray:
        arr = np.asarray(values or [], dtype=np.float32)
        if arr.size == 0:
            return np.zeros(8, dtype=np.float32)
        q25, q50, q75 = np.percentile(arr, [25, 50, 75]).astype(np.float32)
        return np.asarray(
            [
                min(arr.size / 2000.0, 10.0),
                float(arr.mean()),
                float(arr.std()),
                float(arr.min()),
                float(arr.max()),
                float(q25),
                float(q50),
                float(q75),
            ],
            dtype=np.float32,
        )


    def make_graph_stats(n: int, raw_edge_count: int, in_deg: np.ndarray, out_deg: np.ndarray, total_deg: np.ndarray) -> np.ndarray:
        possible_edges = max(1, n * max(1, n - 1))
        density = raw_edge_count / possible_edges
        return np.asarray(
            [
                np.log1p(n) / 10.0,
                np.log1p(raw_edge_count) / 10.0,
                min(density, 1.0),
                float(np.mean(total_deg)) / 20.0,
                float(np.std(total_deg)) / 20.0,
                float(np.max(total_deg)) / 50.0,
                float(np.mean(in_deg)) / 20.0,
                float(np.mean(out_deg)) / 20.0,
            ],
            dtype=np.float32,
        )


    def layer_to_graph(code_id: str, layer: dict) -> GraphData:
        adjacency = layer.get("adjacency", {}) if isinstance(layer, dict) else {}
        raw_nodes = int(adjacency.get("num_nodes", 0) or 0)
        n = max(1, min(raw_nodes, MAX_GRAPH_NODES))
        row = np.asarray(adjacency.get("row", []), dtype=np.int64)
        col = np.asarray(adjacency.get("col", []), dtype=np.int64)
        valid_edges = (row >= 0) & (row < n) & (col >= 0) & (col < n)
        row, col = row[valid_edges], col[valid_edges]
        raw_edge_count = int(min(row.size, col.size))

        if row.size and col.size:
            # Symmetrize directed code graphs for a stable vanilla GCN baseline.
            src = np.concatenate([row, col, np.arange(n, dtype=np.int64)])
            dst = np.concatenate([col, row, np.arange(n, dtype=np.int64)])
        else:
            src = np.arange(n, dtype=np.int64)
            dst = np.arange(n, dtype=np.int64)

        in_deg = np.bincount(dst, minlength=n).astype(np.float32)
        out_deg = np.bincount(src, minlength=n).astype(np.float32)
        total_deg = in_deg + out_deg
        node_pos = np.linspace(0.0, 1.0, n, dtype=np.float32)
        x = np.stack(
            [
                np.ones(n, dtype=np.float32),
                np.log1p(in_deg),
                np.log1p(out_deg),
                np.log1p(total_deg),
                node_pos,
            ],
            axis=1,
        )

        edge_index = np.stack([src, dst], axis=0)
        return GraphData(
            x=torch.from_numpy(x),
            edge_index=torch.from_numpy(edge_index).long(),
            eigen_stats=torch.from_numpy(eigen_stats(layer.get("eigenvalues", []))),
            graph_stats=torch.from_numpy(make_graph_stats(n, raw_edge_count, in_deg, out_deg, total_deg)),
            code_id=code_id,
        )


    def load_graphs_for_type(graph_type: str, needed_ids: set[str]) -> Dict[str, GraphData]:
        graphs: Dict[str, GraphData] = {}
        with open_text(graphs_path) as f:
            for i, line in enumerate(f, start=1):
                if not line.strip():
                    continue
                obj = json.loads(line)
                code_id = str(obj.get("code_id"))
                if code_id not in needed_ids:
                    continue
                layer = obj.get("graphs", {}).get(graph_type, {})
                graphs[code_id] = layer_to_graph(code_id, layer)
                if len(graphs) >= len(needed_ids):
                    break
                if i % 1000 == 0:
                    print(f"  read {i:,} graph rows, loaded {len(graphs):,}/{len(needed_ids):,}")
        missing = len(needed_ids) - len(graphs)
        print(f"{graph_type.upper()}: loaded {len(graphs):,} graphs; missing {missing:,}")
        return graphs


    def filter_pairs_with_graphs(df: pd.DataFrame, graphs: Dict[str, GraphData]) -> pd.DataFrame:
        mask = df.left_id.isin(graphs.keys()) & df.right_id.isin(graphs.keys())
        return df[mask].reset_index(drop=True)


    # =========================
    # Lightweight pure-PyTorch GNN
    # =========================
    def pack_graph_batch(code_ids: List[str], graphs: Dict[str, GraphData], device: str):
        xs = []
        edge_indices = []
        graph_ids = []
        eigen_stats_batch = []
        graph_stats_batch = []
        node_offset = 0

        for graph_idx, code_id in enumerate(code_ids):
            graph = graphs[code_id]
            x = graph.x
            edge_index = graph.edge_index
            n = x.size(0)

            xs.append(x)
            graph_ids.append(torch.full((n,), graph_idx, dtype=torch.long))
            eigen_stats_batch.append(graph.eigen_stats)
            graph_stats_batch.append(graph.graph_stats)
            if edge_index.numel() > 0:
                edge_indices.append(edge_index + node_offset)
            node_offset += n

        x = torch.cat(xs, dim=0).to(device, non_blocking=True)
        graph_batch = torch.cat(graph_ids, dim=0).to(device, non_blocking=True)
        eigen_stats_tensor = torch.stack(eigen_stats_batch, dim=0).to(device, non_blocking=True)
        graph_stats_tensor = torch.stack(graph_stats_batch, dim=0).to(device, non_blocking=True)
        if edge_indices:
            edge_index = torch.cat(edge_indices, dim=1).to(device, non_blocking=True)
        else:
            edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
        return x, edge_index, graph_batch, eigen_stats_tensor, graph_stats_tensor


    class GCNLayer(nn.Module):
        def __init__(self, in_dim: int, out_dim: int):
            super().__init__()
            self.linear = nn.Linear(in_dim, out_dim)

        def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
            src = edge_index[0]
            dst = edge_index[1]
            n = x.size(0)
            h = self.linear(x)
            deg = torch.bincount(dst, minlength=n).float().clamp_min(1.0)
            norm = (deg[src].rsqrt() * deg[dst].rsqrt()).to(h.dtype)
            out = torch.zeros_like(h)
            out.index_add_(0, dst, h[src] * norm.unsqueeze(1))
            return out


    class GraphEncoder(nn.Module):
        def __init__(
            self,
            node_dim: int,
            hidden_dim: int,
            embed_dim: int,
            dropout: float,
            use_eigen_stats: bool,
            use_graph_stats: bool,
        ):
            super().__init__()
            self.gcn1 = GCNLayer(node_dim, hidden_dim)
            self.gcn2 = GCNLayer(hidden_dim, embed_dim)
            self.dropout = dropout
            self.use_eigen_stats = use_eigen_stats
            self.use_graph_stats = use_graph_stats
            self.output_dim = embed_dim
            if use_eigen_stats:
                self.eigen_proj = nn.Sequential(
                    nn.Linear(8, embed_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                    nn.Linear(embed_dim, embed_dim),
                    nn.ReLU(),
                )
                self.output_dim += embed_dim
            if use_graph_stats:
                self.graph_stats_proj = nn.Sequential(
                    nn.Linear(8, embed_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                    nn.Linear(embed_dim, embed_dim),
                    nn.ReLU(),
                )
                self.output_dim += embed_dim

        def forward_packed(
            self,
            x: torch.Tensor,
            edge_index: torch.Tensor,
            graph_batch: torch.Tensor,
            eigen_stats_tensor: torch.Tensor,
            graph_stats_tensor: torch.Tensor,
            num_graphs: int,
        ) -> torch.Tensor:
            h = F.relu(self.gcn1(x, edge_index))
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = F.relu(self.gcn2(h, edge_index))

            pooled = torch.zeros(num_graphs, h.size(1), device=h.device, dtype=h.dtype)
            pooled.index_add_(0, graph_batch, h)
            counts = torch.bincount(graph_batch, minlength=num_graphs).clamp_min(1).to(h.dtype).unsqueeze(1)
            pooled = pooled / counts

            pieces = [pooled]
            if self.use_eigen_stats:
                pieces.append(self.eigen_proj(eigen_stats_tensor))
            if self.use_graph_stats:
                pieces.append(self.graph_stats_proj(graph_stats_tensor))
            return torch.cat(pieces, dim=1)


    class PairGNN(nn.Module):
        def __init__(
            self,
            node_dim: int,
            hidden_dim: int,
            embed_dim: int,
            dropout: float,
            use_eigen_stats: bool,
            use_graph_stats: bool,
        ):
            super().__init__()
            self.encoder = GraphEncoder(node_dim, hidden_dim, embed_dim, dropout, use_eigen_stats, use_graph_stats)
            graph_dim = self.encoder.output_dim
            pair_dim = graph_dim * 4 + 2
            self.classifier = nn.Sequential(
                nn.Linear(pair_dim, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1),
            )

        def forward_pairs(self, left_ids: List[str], right_ids: List[str], graphs: Dict[str, GraphData], device: str) -> torch.Tensor:
            unique_ids = list(dict.fromkeys(list(left_ids) + list(right_ids)))
            x, edge_index, graph_batch, eigen_stats_tensor, graph_stats_tensor = pack_graph_batch(unique_ids, graphs, device)
            emb = self.encoder.forward_packed(
                x,
                edge_index,
                graph_batch,
                eigen_stats_tensor,
                graph_stats_tensor,
                len(unique_ids),
            )

            id_to_pos = {code_id: i for i, code_id in enumerate(unique_ids)}
            left_index = torch.tensor([id_to_pos[code_id] for code_id in left_ids], dtype=torch.long, device=device)
            right_index = torch.tensor([id_to_pos[code_id] for code_id in right_ids], dtype=torch.long, device=device)
            left = emb.index_select(0, left_index)
            right = emb.index_select(0, right_index)
            cosine = F.cosine_similarity(left.float(), right.float(), dim=1).to(left.dtype).unsqueeze(1)
            l2_distance = torch.linalg.vector_norm((left - right).float(), ord=2, dim=1).to(left.dtype).unsqueeze(1)
            feats = torch.cat([left, right, torch.abs(left - right), left * right, cosine, l2_distance], dim=1)
            return self.classifier(feats).squeeze(1)


    # =========================
    # Training and metrics
    # =========================
    from contextlib import nullcontext

    from tqdm.auto import tqdm

    TRAIN_HISTORY = []

    def maybe_autocast(device: str):
        if device == "cuda" and USE_AMP:
            return torch.amp.autocast("cuda")
        return nullcontext()


    def batches(df: pd.DataFrame, batch_size: int, shuffle: bool):
        indices = np.arange(len(df))
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, len(indices), batch_size):
            yield df.iloc[indices[start:start + batch_size]]


    def binary_metrics(labels: np.ndarray, probs: np.ndarray, threshold: float = 0.5) -> dict:
        preds = (probs >= threshold).astype(np.int64)
        labels = labels.astype(np.int64)
        tp = int(((preds == 1) & (labels == 1)).sum())
        fp = int(((preds == 1) & (labels == 0)).sum())
        tn = int(((preds == 0) & (labels == 0)).sum())
        fn = int(((preds == 0) & (labels == 1)).sum())
        precision = tp / max(1, tp + fp)
        recall = tp / max(1, tp + fn)
        f1 = 2 * precision * recall / max(1e-12, precision + recall)
        specificity = tn / max(1, tn + fp)
        negative_precision = tn / max(1, tn + fn)
        negative_f1 = 2 * negative_precision * specificity / max(1e-12, negative_precision + specificity)
        acc = (tp + tn) / max(1, len(labels))
        return {
            "P": precision, "R": recall, "F1": f1, "Acc": acc,
            "MacroF1": 0.5 * (f1 + negative_f1),
            "BalancedAccuracy": 0.5 * (recall + specificity),
            "ROC_AUC": float(roc_auc_score(labels, probs)) if len(np.unique(labels)) == 2 else float("nan"),
            "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        }


    def gnn_is_cuda_oom(error: BaseException) -> bool:
        return isinstance(error, torch.OutOfMemoryError) or (
            isinstance(error, RuntimeError) and "out of memory" in str(error).lower()
        )


    @torch.no_grad()
    def predict_batch_adaptive(model: PairGNN, batch: pd.DataFrame, graphs: Dict[str, GraphData], device: str) -> np.ndarray:
        logits = None
        try:
            with maybe_autocast(device):
                logits = model.forward_pairs(batch.left_id.tolist(), batch.right_id.tolist(), graphs, device)
            return torch.sigmoid(logits).float().cpu().numpy()
        except RuntimeError as error:
            if not gnn_is_cuda_oom(error) or len(batch) <= 1:
                raise
            logits = None
            del error
            gc.collect()
            torch.cuda.empty_cache()
            middle = len(batch) // 2
            return np.concatenate((
                predict_batch_adaptive(model, batch.iloc[:middle], graphs, device),
                predict_batch_adaptive(model, batch.iloc[middle:], graphs, device),
            ))


    @torch.no_grad()
    def predict_probs(model: PairGNN, df: pd.DataFrame, graphs: Dict[str, GraphData], device: str) -> tuple[np.ndarray, np.ndarray]:
        model.eval()
        all_probs = []
        all_labels = []
        for batch in batches(df, BATCH_SIZE, shuffle=False):
            all_probs.append(predict_batch_adaptive(model, batch, graphs, device))
            all_labels.append(batch.label.to_numpy(dtype=np.int64))
        probs = np.concatenate(all_probs) if all_probs else np.asarray([], dtype=np.float32)
        labels = np.concatenate(all_labels) if all_labels else np.asarray([], dtype=np.int64)
        return probs, labels


    @torch.no_grad()
    def evaluate(model: PairGNN, df: pd.DataFrame, graphs: Dict[str, GraphData], device: str, threshold: float) -> dict:
        probs, labels = predict_probs(model, df, graphs, device)
        return binary_metrics(labels, probs, threshold=threshold)


    def print_split_baselines(name: str, df: pd.DataFrame) -> None:
        labels = df.label.to_numpy(dtype=np.int64)
        pos_rate = float(labels.mean()) if labels.size else 0.0
        all_positive = binary_metrics(labels, np.ones_like(labels, dtype=np.float32), threshold=0.5)
        all_negative = binary_metrics(labels, np.zeros_like(labels, dtype=np.float32), threshold=0.5)
        print(
            f"{name} positive_rate={pos_rate:.4f} "
            f"all_pos_f1={all_positive['F1']:.4f} all_pos_acc={all_positive['Acc']:.4f} "
            f"all_neg_f1={all_negative['F1']:.4f} all_neg_acc={all_negative['Acc']:.4f}"
        )


    def validation_selection_context(labels: np.ndarray) -> dict:
        labels = np.asarray(labels, dtype=np.int64)
        positives = int((labels == 1).sum())
        negatives = int((labels == 0).sum())
        balanced = positives > 0 and positives == negatives
        return {"balanced": balanced, "metric": "Accuracy" if balanced else "F1", "positives": positives, "negatives": negatives}


    def validation_selection_key(metrics: dict, labels: np.ndarray) -> tuple[float, float]:
        context = validation_selection_context(labels)
        return (metrics["Acc"], metrics["F1"]) if context["balanced"] else (metrics["F1"], metrics["BalancedAccuracy"])


    def best_validation_threshold(labels: np.ndarray, probs: np.ndarray) -> tuple[float, dict]:
        if probs.size == 0:
            return 0.5, binary_metrics(labels, probs, threshold=0.5)
        candidates = np.unique(np.quantile(probs, np.linspace(0.0, 1.0, 101)))
        candidates = np.unique(np.concatenate([candidates, np.asarray([0.5], dtype=np.float32)]))
        best_threshold = 0.5
        best_metrics = None
        for threshold in candidates:
            metrics = binary_metrics(labels, probs, threshold=float(threshold))
            if best_metrics is None or validation_selection_key(metrics, labels) > validation_selection_key(best_metrics, labels):
                best_metrics = metrics
                best_threshold = float(threshold)
        return best_threshold, best_metrics or binary_metrics(labels, probs, threshold=0.5)


    def train_one_graph_type(graph_type: str) -> dict:
        graph_started = time.perf_counter()
        print("\n" + "=" * 80)
        print("Graph type:", graph_type.upper())
        graphs = load_graphs_for_type(graph_type, selected_ids)

        tr = filter_pairs_with_graphs(train_pairs, graphs)
        va = filter_pairs_with_graphs(valid_pairs, graphs)
        te = filter_pairs_with_graphs(test_pairs, graphs)
        print("usable train/valid/test:", len(tr), len(va), len(te))
        print_split_baselines("train", tr)
        if len(va):
            print_split_baselines("valid", va)
        print_split_baselines("test", te)
        if len(tr) == 0 or len(te) == 0:
            return {
                "Method": graph_type.upper(), "P": 0.0, "R": 0.0, "F1": 0.0, "Acc": 0.0,
                "MacroF1": 0.0, "BalancedAccuracy": 0.0, "ROC_AUC": float("nan"),
                "TrainableParameters": 0, "RuntimeSeconds": float(time.perf_counter() - graph_started),
                "RuntimeMinutes": float(time.perf_counter() - graph_started) / 60.0,
            }

        model = PairGNN(
            node_dim=5,
            hidden_dim=HIDDEN_DIM,
            embed_dim=EMBED_DIM,
            dropout=DROPOUT,
            use_eigen_stats=USE_EIGEN_STATS,
            use_graph_stats=USE_GRAPH_STATS,
        ).to(DEVICE)

        positives = max(1, int((tr.label == 1).sum()))
        negatives = max(1, int((tr.label == 0).sum()))
        pos_weight = torch.tensor([negatives / positives], dtype=torch.float32, device=DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
        amp_enabled = DEVICE == "cuda" and USE_AMP
        scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

        best_state = None
        best_valid_f1 = -1.0
        best_selection_key = None
        best_epoch = 0
        best_threshold = THRESHOLD
        stale_epochs = 0

        training_microbatch = BATCH_SIZE
        for epoch in range(1, EPOCHS + 1):
            model.train()
            losses = []
            train_batches = batches(tr, BATCH_SIZE, shuffle=True)
            total_batches = math.ceil(len(tr) / BATCH_SIZE)
            train_progress = tqdm(
                train_batches,
                total=total_batches,
                desc=f"{graph_type.upper()} epoch {epoch:02d}/{EPOCHS}",
                unit="batch",
                leave=False,
            )
            for batch in train_progress:
                while True:
                    optimizer.zero_grad(set_to_none=True)
                    weighted_loss = 0.0
                    micro = labels = logits = loss = scaled_loss = None
                    try:
                        for start in range(0, len(batch), training_microbatch):
                            micro = batch.iloc[start:start + training_microbatch]
                            labels = torch.tensor(micro.label.to_numpy(dtype=np.float32), device=DEVICE)
                            with maybe_autocast(DEVICE):
                                logits = model.forward_pairs(
                                    micro.left_id.tolist(), micro.right_id.tolist(), graphs, DEVICE
                                )
                                loss = criterion(logits, labels)
                                scaled_loss = loss * (len(micro) / len(batch))
                            scaler.scale(scaled_loss).backward()
                            weighted_loss += float(loss.detach().cpu()) * len(micro)
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                        scaler.step(optimizer)
                        scaler.update()
                        losses.append(weighted_loss / max(1, len(batch)))
                        break
                    except RuntimeError as error:
                        if not gnn_is_cuda_oom(error) or training_microbatch <= 1:
                            raise
                        previous = training_microbatch
                        training_microbatch = max(1, training_microbatch // 2)
                        optimizer.zero_grad(set_to_none=True)
                        micro = labels = logits = loss = scaled_loss = None
                        del error
                        gc.collect()
                        torch.cuda.empty_cache()
                        print(
                            f"[CUDA OOM recovery] {graph_type.upper()} microbatch "
                            f"{previous} -> {training_microbatch}"
                        )
                if losses:
                    train_progress.set_postfix(
                        loss=f"{np.mean(losses[-50:]):.4f}", microbatch=training_microbatch
                    )

            eval_df = va if len(va) else tr
            valid_probs, valid_labels = predict_probs(model, eval_df, graphs, DEVICE)
            if TUNE_THRESHOLD_ON_VALID:
                epoch_threshold, valid_metrics = best_validation_threshold(valid_labels, valid_probs)
            else:
                epoch_threshold = THRESHOLD
                valid_metrics = binary_metrics(valid_labels, valid_probs, threshold=THRESHOLD)
            mean_loss = float(np.mean(losses)) if losses else 0.0
            current_lr = float(optimizer.param_groups[0]["lr"])
            current_selection_key = validation_selection_key(valid_metrics, valid_labels)
            scheduler.step(current_selection_key[0])
            TRAIN_HISTORY.append({
                "Method": graph_type.upper(),
                "epoch": epoch,
                "train_loss": mean_loss,
                "valid_P": valid_metrics["P"],
                "valid_R": valid_metrics["R"],
                "valid_F1": valid_metrics["F1"],
                "valid_Acc": valid_metrics["Acc"],
                "threshold": epoch_threshold,
                "lr": current_lr,
            })
            print(
                f"epoch {epoch:02d} loss={mean_loss:.4f} "
                f"valid_f1={valid_metrics['F1']:.4f} valid_acc={valid_metrics['Acc']:.4f} "
                f"thr={epoch_threshold:.4f} lr={current_lr:.2e}"
            )
            if best_selection_key is None or current_selection_key > best_selection_key:
                best_selection_key = current_selection_key
                best_valid_f1 = valid_metrics["F1"]
                best_epoch = epoch
                best_threshold = epoch_threshold
                stale_epochs = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                stale_epochs += 1
                if stale_epochs >= PATIENCE:
                    print(f"early stopping at epoch {epoch}; best_epoch={best_epoch} best_valid_f1={best_valid_f1:.4f}")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        eval_df = va if len(va) else tr
        valid_probs, valid_labels = predict_probs(model, eval_df, graphs, DEVICE)
        if TUNE_THRESHOLD_ON_VALID:
            selected_threshold, selected_valid_metrics = best_validation_threshold(valid_labels, valid_probs)
        else:
            selected_threshold = THRESHOLD
            selected_valid_metrics = binary_metrics(valid_labels, valid_probs, threshold=THRESHOLD)
        print(f"selected threshold={selected_threshold:.4f} selected_valid_f1={selected_valid_metrics['F1']:.4f}")
        test_probs, test_labels = predict_probs(model, te, graphs, DEVICE)
        test_metrics = binary_metrics(test_labels, test_probs, threshold=selected_threshold)
        record_language_breakdown(te, test_probs, selected_threshold, dataset=DATASET_KEY_FOR_BREAKDOWN, method=f"GNN-{graph_type.upper()}", graph_type=graph_type)
        return {
            "Method": graph_type.upper(),
            "BestEpoch": best_epoch,
            "BestValidF1": best_valid_f1,
            "BestValidAcc": selected_valid_metrics["Acc"],
            "ValidationSelectionMetric": validation_selection_context(valid_labels)["metric"],
            "ValidationBalanced": validation_selection_context(valid_labels)["balanced"],
            "ValidationPositives": validation_selection_context(valid_labels)["positives"],
            "ValidationNegatives": validation_selection_context(valid_labels)["negatives"],
            "P": test_metrics["P"],
            "R": test_metrics["R"],
            "F1": test_metrics["F1"],
            "Acc": test_metrics["Acc"],
            "MacroF1": test_metrics["MacroF1"],
            "BalancedAccuracy": test_metrics["BalancedAccuracy"],
            "ROC_AUC": test_metrics["ROC_AUC"],
            "Threshold": selected_threshold,
            "TP": test_metrics["TP"],
            "FP": test_metrics["FP"],
            "TN": test_metrics["TN"],
            "FN": test_metrics["FN"],
            "TrainPairs": len(tr),
            "ValidPairs": len(eval_df),
            "TestPairs": len(te),
            "TrainableParameters": int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)),
            "MinimumTrainingMicrobatch": int(training_microbatch),
            "RuntimeSeconds": float(time.perf_counter() - graph_started),
            "RuntimeMinutes": float(time.perf_counter() - graph_started) / 60.0,
        }


    # =========================
    # Run all GNN baselines
    # =========================
    TRAIN_HISTORY.clear()
    results = []
    graph_progress = tqdm(GRAPH_TYPES, desc="GNN graph baselines", unit="graph")
    for graph_type in graph_progress:
        graph_progress.set_postfix(graph=graph_type.upper())
        result = train_one_graph_type(graph_type)
        results.append(result)
        graph_progress.set_postfix(graph=graph_type.upper(), f1=f"{result.get('F1', 0.0):.4f}")
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    results_df = pd.DataFrame(results)
    score_cols = ["P", "R", "F1", "Acc", "MacroF1", "BalancedAccuracy", "ROC_AUC"]
    for col in score_cols:
        results_df[col] = results_df[col].astype(float)

    print("\nFinal results")
    display(results_df[["Method", "P", "R", "F1", "Acc", "MacroF1", "BalancedAccuracy", "ROC_AUC", "TrainableParameters", "RuntimeSeconds", "RuntimeMinutes"]].style.format({c: "{:.4f}" for c in score_cols}))

    out_path = Path(f"/kaggle/working/{DATASET_KEY}_gnn_baseline_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    history_df = pd.DataFrame(TRAIN_HISTORY)
    history_path = Path(f"/kaggle/working/{DATASET_KEY}_gnn_training_history.csv")
    history_df.to_csv(history_path, index=False)
    print("Saved:", history_path)


    # =========================
    # Plot GNN loss after training
    # =========================
    import matplotlib.pyplot as plt

    history_path = Path(f"/kaggle/working/{DATASET_KEY}_gnn_training_history.csv")
    if "history_df" not in globals() or history_df.empty:
        history_df = pd.read_csv(history_path)

    print("History rows:", len(history_df))
    display(history_df)

    method_col = "Method" if "Method" in history_df.columns else None

    plt.figure(figsize=(10, 4))
    if method_col:
        for method, part in history_df.groupby(method_col):
            plt.plot(part["epoch"], part["train_loss"], marker="o", label=method)
        plt.legend(loc="best")
    else:
        plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
    plt.title("GNN train loss")
    plt.xlabel("Epoch")
    plt.ylabel("BCE loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    loss_plot_path = Path(f"/kaggle/working/{DATASET_KEY}_gnn_loss_curve.png")
    plt.savefig(loss_plot_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", loss_plot_path)

    plt.figure(figsize=(10, 4))
    if method_col:
        for method, part in history_df.groupby(method_col):
            plt.plot(part["epoch"], part["valid_F1"], marker="o", label=f"{method} F1")
        plt.legend(loc="best")
    else:
        plt.plot(history_df["epoch"], history_df["valid_F1"], marker="o", label="Valid F1")
        plt.legend(loc="best")
    plt.title("GNN validation F1")
    plt.xlabel("Epoch")
    plt.ylabel("F1")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    valid_plot_path = Path(f"/kaggle/working/{DATASET_KEY}_gnn_valid_f1_curve.png")
    plt.savefig(valid_plot_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", valid_plot_path)

    plt.figure(figsize=(10, 4))
    if method_col:
        for method, part in history_df.groupby(method_col):
            plt.plot(part["epoch"], part["valid_Acc"], marker="o", label=f"{method} Acc")
        plt.legend(loc="best")
    else:
        plt.plot(history_df["epoch"], history_df["valid_Acc"], marker="o", label="Valid Acc")
        plt.legend(loc="best")
    plt.title("GNN validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    acc_plot_path = Path(f"/kaggle/working/{DATASET_KEY}_gnn_valid_acc_curve.png")
    plt.savefig(acc_plot_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", acc_plot_path)

    if "lr" in history_df.columns:
        plt.figure(figsize=(10, 3))
        if method_col:
            for method, part in history_df.groupby(method_col):
                plt.plot(part["epoch"], part["lr"], marker="o", label=method)
            plt.legend(loc="best")
        else:
            plt.plot(history_df["epoch"], history_df["lr"], marker="o")
        plt.title("Learning rate")
        plt.xlabel("Epoch")
        plt.ylabel("LR")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    # Research-reproducibility manifest and enriched result table.
    # This is deliberately written after evaluation so measured runtime is final.
    import datetime as _datetime
    try:
        _gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        _gpu_capability = ".".join(map(str, torch.cuda.get_device_capability(0))) if torch.cuda.is_available() else None
        _torch_version = torch.__version__
    except Exception:
        _gpu_name, _gpu_capability, _torch_version = "unavailable", None, "unavailable"
    _completed_utc = _datetime.datetime.now(_datetime.timezone.utc).isoformat()
    _run_seconds = float(time.perf_counter() - run_started)
    _shared_fields = {
        "Dataset": DATASET_KEY,
        "RunProfile": RUN_PROFILE,
        "Seed": int(SEED),
        "ConfiguredEpochs": int(EPOCHS) if "EPOCHS" in locals() else None,
        "BatchSize": int(BATCH_SIZE) if "BATCH_SIZE" in locals() else None,
        "LearningRate": float(LEARNING_RATE) if "LEARNING_RATE" in locals() else None,
        "WeightDecay": float(WEIGHT_DECAY) if "WEIGHT_DECAY" in locals() else None,
        "GPU": _gpu_name,
        "GPUCapability": _gpu_capability,
        "TorchVersion": _torch_version,
        "CompletedUTC": _completed_utc,
    }
    if "results_df" in locals():
        _result_table = results_df.copy()
    elif "result" in locals():
        _result_table = pd.DataFrame([result])
    elif "row" in locals():
        _result_table = pd.DataFrame([row])
    else:
        _result_table = pd.DataFrame()
    for _field, _value in _shared_fields.items():
        _result_table[_field] = _value
    if "RuntimeSeconds" not in _result_table.columns:
        _result_table["RuntimeSeconds"] = _run_seconds
    if "RuntimeMinutes" not in _result_table.columns:
        _result_table["RuntimeMinutes"] = _run_seconds / 60.0
    _result_path = RESULTS_PATH if "RESULTS_PATH" in locals() else out_path
    _result_table.to_csv(_result_path, index=False)
    _metadata = {
        **_shared_fields,
        "RunLabel": RUN_LABEL,
        "RuntimeSeconds": _run_seconds,
        "RuntimeMinutes": _run_seconds / 60.0,
        "RequestedPairCaps": {
            "train": MAX_TRAIN_PAIRS,
            "valid": MAX_VALID_PAIRS,
            "test": MAX_TEST_PAIRS,
        },
        "ModelConfiguration": {
            _name: locals().get(_name)
            for _name in (
                "MAX_AST_NODES", "MAX_AST_EDGES", "MAX_STATEMENTS", "MAX_NODE_TYPES", "MAX_NODES",
                "EMBED_DIM", "HIDDEN_DIM", "TREE_HIDDEN_DIM", "CODE_DIM", "DROPOUT",
                "GRAPH_TYPE", "GRAPH_TYPES", "K_EIGEN", "USE_EIGEN_STATS", "USE_GRAPH_STATS",
            ) if _name in locals()
        },
        "OutputFiles": {
            "results": str(_result_path),
            "history": str(HISTORY_PATH) if "HISTORY_PATH" in locals() else (str(history_path) if "history_path" in locals() else None),
        },
    }
    _metadata_path = WORK_DIR / f"{DATASET_KEY}_{RUN_LABEL}_run_metadata.json"
    _metadata_path.write_text(json.dumps(_metadata, indent=2, default=str), encoding="utf-8")
    print("Research metadata:", _metadata_path)
    if "results_df" in locals():
        results_df = _result_table

    if "results_df" in locals():
        return results_df.copy()
    if "result" in locals():
        return pd.DataFrame([result])
    if "row" in locals():
        return pd.DataFrame([row])
    raise RuntimeError("The baseline did not produce a result table.")


from IPython.display import display
import pandas as pd

all_dataset_results = {}
for current_dataset_key in DATASET_KEYS:
    print("\n" + "=" * 96)
    print(f"Running {current_dataset_key.upper()}")
    print("=" * 96)
    dataset_results = run_one_dataset(current_dataset_key)
    # The per-run result table already carries Dataset for research metadata.
    # Preserve a single authoritative value rather than inserting a duplicate column.
    if "Dataset" in dataset_results.columns:
        dataset_results["Dataset"] = current_dataset_key.upper()
    else:
        dataset_results.insert(0, "Dataset", current_dataset_key.upper())
    all_dataset_results[current_dataset_key] = dataset_results

print("\n" + "=" * 96)
print("Final result tables")
print("=" * 96)
for current_dataset_key in DATASET_KEYS:
    print(f"\n{current_dataset_key.upper()} results")
    display(all_dataset_results[current_dataset_key])

combined_results = pd.concat(
    [all_dataset_results[key] for key in DATASET_KEYS],
    ignore_index=True,
)
combined_path = Path("/kaggle/working") / f"{RUN_LABEL}_combined_dataset_results.csv"
combined_results.to_csv(combined_path, index=False)
print("Combined results:", combined_path)
